"""Grid search job for tuning ALS fits on different models."""

In [0]:
%run ../../config/utils

In [0]:
from datetime import datetime
from dateutil import parser
import time
import os
import sys
sys.path.append('..')
sys.path.append('../..')


from pyspark.ml.recommendation import ALS
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col
from pyspark.sql import Row

import mlflow
from mlflow.models.signature import infer_signature
from mlflow.client import MlflowClient

from lib_cf.cf_io import (
    load_config,
    calculate_filepaths,
)
from lib_cf.models import evaluate_cf_train
from lib_cf.models import (
    scale_by_group,
    purchase_cycle_normalize,
    cat_size_normalize,
)

In [0]:
config_path = dbutils.widgets.get("config_path")
test = False if not dbutils.widgets.get("test") else True
rmse = False if not dbutils.widgets.get("rmse") else True
future = False if not dbutils.widgets.get("future") else True

In [0]:
# (1) ---- ARGUMENTS ---- #
CNF, CFG_PATH = load_config(config_path, test, future, rmse)
PARAMS = dict(
    list(CNF["shared"].items())
    + list(CNF["grid_search"].items())
    + list(CNF["train"].items())
)
PATHS = CNF["paths"]
PARAMS, PATHS = calculate_filepaths(PARAMS, PATHS)

start_dt = parser.parse(PARAMS["start"])
end_dt = parser.parse(PARAMS["end"])
time_period = (end_dt - start_dt).days

In [0]:
experiment_name = experiment_name_cf_model_grid_search

mlflow.spark.autolog()

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri('databricks-uc')

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name) 
mlflow.set_experiment(experiment_name)

mark_datetime = datetime.strftime(datetime.now(), '_%Y-%m-%d_%H-%M-%S')
run = mlflow.start_run(run_name=f'training_{mark_datetime}')

# feature_dataset = mlflow.data.from_spark(spark.table(model_trip_spend_etl_output).select(*column_headers), name = 'trips_spend_etl_dataset')

In [0]:
output = []
for reg in PARAMS["lambdas"]:
    for alpha in PARAMS["alphas"]:
        for iteration in PARAMS["iters"]:
            for rank in PARAMS["ranks"]:
                for binary in PARAMS["binarys"]:
                    for mem_norm in PARAMS["normalizes"]:
                        for cat_norm in PARAMS["normalizes"]:
                            for size_norm in [False]:
                                for inputtype in PARAMS["inputtypes"]:
                                    payload = {
                                        "mlflow_run_id": run.info.run_id,
                                        "category": PARAMS["category"],
                                        "start_date": start_dt.date(),
                                        "end_date": end_dt.date(),
                                        "run_date": datetime.now().date(),
                                        "regParam": float(reg),
                                        "alpha": float(alpha),
                                        "maxIter": int(iteration),
                                        "rank": int(rank),
                                        "binary": bool(binary),
                                        "member_norm": bool(mem_norm),
                                        "cat_norm": bool(cat_norm),
                                        "size_norm": bool(size_norm),
                                        "inputtype": inputtype,
                                    }
                                    print(f"training param set... \n{payload}")
                                    data = read_cf_tables(cf_matrix, PARAMS)
                                    data = data.select(
                                        "MBRSHP_SID", "CATEGORY_ID", inputtype
                                    )
                                    feature_dataset = mlflow.data.from_spark(data, name = 'cf_etl_dataset')
                                    with mlflow.start_run(nested=True) as run_nested:
                                        payload['mlflow_nested_run_id'] = run_nested.info.run_id
                                        mlflow.log_input(feature_dataset, context="source")
                                        PARAMS['inputtype'] = inputtype
                                        if binary:
                                            condition = data[inputtype] > 0
                                            data = data.withColumn(
                                                inputtype,
                                                when(condition, 1).otherwise(0),
                                            )
                                        if cat_norm:
                                            cat_dna = spark.table(globals()[f"fs_{PARAMS['category'].lower()}_cd"])
                                            data = purchase_cycle_normalize(
                                                PARAMS,
                                                cat_dna,
                                                data,
                                                time_period,
                                            )
                                        if mem_norm:
                                            data = scale_by_group(
                                                data,
                                                PARAMS["inputtype"],
                                                "MBRSHP_SID",
                                                "mean",
                                            )
                                        if size_norm:
                                            item_brand = spark.table(silver_ad_hoc_master_item_with_brand)
                                            data = cat_size_normalize(
                                                PARAMS, item_brand, data
                                            )
                                        data.cache()
                                        als = ALS(
                                            maxIter=iteration,
                                            rank=rank,  # depth of vector to learn
                                            regParam=reg,  # lambda
                                            alpha=alpha,  # preference alpha
                                            userCol="MBRSHP_SID",
                                            itemCol="CATEGORY_ID",
                                            numUserBlocks=10,  # 'partitions' of members
                                            numItemBlocks=10,  # 'partitions' of categories
                                            checkpointInterval=10,  # interval to checkpoint cache()
                                            nonnegative=True,  # nonnegative sets most results to 0
                                            ratingCol=inputtype,
                                            implicitPrefs=True,
                                            coldStartStrategy="drop",
                                            # intermediateStorageLevel="DISK_ONLY",
                                            # finalStorageLevel="DISK_ONLY",
                                        )

                                        model = als.fit(data)

                                        add_params = {
                                            "userCol": "MBRSHP_SID",
                                            "itemCol": "CATEGORY_ID",
                                            "ratingCol": inputtype,
                                            "implicitPrefs": als.getImplicitPrefs(),
                                            "nonnegative": als.getNonnegative(),
                                            "coldStartStrategy": als.getColdStartStrategy(),
                                            "numUserBlocks": als.getNumUserBlocks(),
                                            "numItemBlocks": als.getNumItemBlocks(),
                                        }

                                        payload = {**payload, **add_params}

                                        mlflow.log_params(payload)

                                        print("evaluating param set...")

                                        slate = read_cf_tables(cf_slate, PARAMS)
                                        pdata = read_cf_tables(cf_matrix, PARAMS, past_flag=True)
                                        fdata = read_cf_tables(cf_matrix, PARAMS, future_flag=True)
                                        cat_lookup = read_cf_tables(cf_cat_lookup, PARAMS)
                                        evals = evaluate_cf_train(
                                            spark,
                                            model,
                                            pdata,
                                            slate,
                                            fdata,
                                            cat_lookup,
                                            eval_col=PARAMS["eval_col"],
                                            backtest=PARAMS["backtest"],
                                            level=PARAMS["category"],
                                            params=PARAMS,
                                        )
                                        top_cat = evals.pop('top_cat', None)
                                        evals.pop('backtest', None)
                                        mlflow.log_metrics(evals)
                                        mlflow.log_param("eval_top_cat", top_cat)

                                        mlflow.log_artifact(config_path)
                                        
                                        payload = {**payload, **evals}
                                        payload['top_cat'] = top_cat

                                        df_row = spark.createDataFrame([Row(**payload)])
                                        df_row.write.mode('append').option('mergeSchema', 'true').saveAsTable(cf_model_grid_search)
                                        
                                        # log_line = {
                                        #     "run_name": PARAMS["run_name"],
                                        #     "date": datetime.now().strftime(
                                        #         "%Y%m%d"
                                        #     ),
                                        #     "data": PARAMS["data"],
                                        #     "future": PARAMS["future"],
                                        #     "data_version": PARAMS["data_version"],
                                        #     "model_version": PARAMS[
                                        #         "model_version"
                                        #     ],
                                        #     "category": PARAMS["category"],
                                        #     "num_cats": PARAMS["num_cats"],
                                        #     "binary": binary,
                                        #     "backtest": PARAMS["backtest"],
                                        #     "member_norm": mem_norm,
                                        #     "cat_norm": cat_norm,
                                        #     "size_norm": size_norm,
                                        #     "inputtype": inputtype,
                                        #     "eval_col": PARAMS["eval_col"],
                                        #     "rank": rank,
                                        #     "lambda": reg,
                                        #     "alpha": alpha,
                                        #     "iter": iteration,
                                        #     "backtest": evals["backtest"],
                                        #     "mean_score": evals["mean_score"],
                                        #     "count_under05": evals[
                                        #         "count_under05"
                                        #     ],
                                        #     "top_cat": evals["top_cat"],
                                        #     "top_cat_ct": evals["top_cat_ct"],
                                        #     "rmse": evals["rmse"],
                                        #     "hits_1": evals["hits_1"],
                                        #     "hits_05": evals["hits_05"],
                                        #     "overall": evals["overall"],
                                        #     "personal": evals["personal"],
                                        #     "LUT": evals["lut"],
                                        #     "cnfg_file": CNFG_OUTPUT_PATH,
                                        # }

In [0]:
mlflow.end_run()